# 第 9 章 · 预训练 Pretrain

> 📄 配套动画网页：`../09_pretrain.html`　（先用浏览器打开，把原理动画看一遍再来写代码）

**所属部分**：第三部分 · 训练

模型写好了，现在让它真正开始学习。预训练就是在大规模文本上反复做「下一词预测」。

---


## 🎯 本章目标

- 掌握 `optim.AdamW`、`GradScaler`、`autocast`、`clip_grad_norm_`
- 理解梯度累积与有效 batch size 的关系
- 能实现余弦学习率调度
- 能写出完整的训练循环 `train_epoch`

---


## 📖 背景：训练循环的五个动作

```python
for input_ids, labels in loader:
    with autocast():                     # 1. 混合精度前向
        loss = model(input_ids, labels=labels).loss / accumulation_steps
    scaler.scale(loss).backward()        # 2. 反向传播
    scaler.unscale_(optimizer)
    clip_grad_norm_(model.parameters(), 1.0)   # 3. 梯度裁剪
    scaler.step(optimizer); scaler.update()    # 4. 更新参数
    optimizer.zero_grad(set_to_none=True)      # 5. 清空梯度
```

**梯度累积**：显存不够放不下大 batch 时，把多个小 batch 的梯度攒起来再更新一次，
等效于大 batch。注意 loss 要先除以累积步数。

**余弦学习率**：MiniMind 用 `lr_t = lr · (0.1 + 0.45·(1 + cos(π·step/total)))`。

---



## 🖼 官方配图

![MiniMind 预训练损失曲线——loss 稳定下降即训练正常](../images/pretrain_loss.jpg)

*MiniMind 预训练损失曲线——loss 稳定下降即训练正常*

---


# 第一部分 · 基础函数演示

> 这一部分**可以直接运行**。先把本章要用到的 Python / PyTorch 函数和方法摸熟，后面自己写的时候才不会卡在 API 上。


### 9.1 `optim.AdamW`

In [ ]:
import torch
from torch import nn, optim

model = nn.Linear(4, 4)
opt = optim.AdamW(model.parameters(), lr=1e-3)

loss = model(torch.randn(2, 4)).sum()
loss.backward()
print("更新前 weight[0,0]:", model.weight[0, 0].item())
opt.step()
opt.zero_grad(set_to_none=True)
print("更新后 weight[0,0]:", model.weight[0, 0].item())
print("梯度已清空:", model.weight.grad is None)

### 9.2 梯度裁剪：`clip_grad_norm_`

梯度突然变得很大（梯度爆炸）是训练崩溃的常见原因。裁剪把梯度的总范数限制在阈值内。

In [ ]:
model = nn.Linear(4, 4)
model.weight.grad = torch.full_like(model.weight, 100.0)   # 人为造一个巨大的梯度
norm_before = model.weight.grad.norm().item()
total = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
print(f"裁剪前范数: {norm_before:.1f}  裁剪后范数: {model.weight.grad.norm().item():.3f}")

### 9.3 混合精度：`autocast` + `GradScaler`

用 bf16/fp16 计算，显存减半、速度更快。`GradScaler` 负责在 fp16 下防止梯度下溢。

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = nn.Linear(8, 8).to(device)
scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))
opt = optim.AdamW(model.parameters(), lr=1e-3)

x = torch.randn(4, 8, device=device)
with torch.cuda.amp.autocast(enabled=(device == "cuda"), dtype=torch.bfloat16):
    loss = model(x).pow(2).mean()
scaler.scale(loss).backward()
scaler.step(opt)
scaler.update()
print(f"混合精度前向成功，loss={loss.item():.4f}（设备 {device}）")

### 9.4 余弦学习率调度

In [ ]:
import math

def get_lr(current_step, total_steps, lr):
    return lr * (0.1 + 0.45 * (1 + math.cos(math.pi * current_step / total_steps)))

total = 1000
lrs = [get_lr(s, total, 5e-4) for s in range(total)]
print(f"初始 lr = {lrs[0]:.6f}   结束 lr = {lrs[-1]:.6f}   比值 = {lrs[-1]/lrs[0]:.2f}")

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6, 3))
    plt.plot(lrs)
    plt.xlabel("step"); plt.ylabel("lr"); plt.title("Cosine LR Schedule")
    plt.grid(alpha=.3); plt.tight_layout(); plt.show()
except ImportError:
    print("未安装 matplotlib（`uv sync` 会装上）。用文本方式看曲线形状：")
    for i in range(0, total, 100):
        bar = "█" * int(lrs[i] / max(lrs) * 40)
        print(f"  step {i:4d}: {bar} {lrs[i]:.6f}")

### 9.5 梯度累积

累积 `n` 步再更新，等效 batch = `batch_size × n`。

In [ ]:
model = nn.Linear(4, 4)
opt = optim.AdamW(model.parameters(), lr=1e-2)
accumulation_steps = 4
before = model.weight[0, 0].item()

for i in range(accumulation_steps):
    loss = model(torch.randn(2, 4)).pow(2).mean() / accumulation_steps
    loss.backward()
    if (i + 1) % accumulation_steps == 0:
        opt.step(); opt.zero_grad(set_to_none=True)

print("参数已更新:", before != model.weight[0, 0].item())

# 第二部分 · 动手实现（只给提示，留白给你）

> 下面每一节都**没有答案**。请根据文字提示 + 第一部分演示过的函数，自己从零写出来。
> 写不出来时，先回去看第一部分的演示，再对照网页里的动画想清楚数据形状。

现在写出完整的训练循环。

> 💡 每个留白代码块后面都跟了一个**测试块**。先写代码，再运行测试块。


In [ ]:
# ════════════════════ 自测工具 ════════════════════
# 每个留白代码块后面都跟一个测试块。先在上面写代码，再运行测试块。
# 没写之前运行也不会报错，只会提示「未实现」。
import torch
import torch.nn.functional as F

_PASS, _FAIL, _SKIP = [], [], []

def check(name, fn):
    """运行一个测试。捕获异常并打印结果，不会中断后面的测试。"""
    try:
        fn()
        print(f"✅ 通过   {name}")
        _PASS.append(name)
    except NameError as e:
        print(f"⏸  未实现 {name} —— {e}")
        _SKIP.append(name)
    except AssertionError as e:
        print(f"❌ 失败   {name} —— {e}")
        _FAIL.append(name)
    except Exception as e:
        print(f"❌ 出错   {name} —— {type(e).__name__}: {e}")
        _FAIL.append(name)

def close(a, b, tol=1e-4, msg=""):
    """断言两个张量 / 数字足够接近"""
    a, b = torch.as_tensor(a).float().reshape(-1), torch.as_tensor(b).float().reshape(-1)
    assert a.numel() == b.numel(), f"元素个数不一致 {a.numel()} vs {b.numel()} {msg}"
    assert torch.allclose(a, b, atol=tol), f"数值不接近（容差 {tol}）{msg}"

def same(a, b, msg=""):
    """断言两个张量完全相同"""
    a, b = torch.as_tensor(a), torch.as_tensor(b)
    assert a.shape == b.shape, f"形状不一致 {tuple(a.shape)} vs {tuple(b.shape)} {msg}"
    assert torch.equal(a, b), f"内容不相同 {msg}"

def summary():
    print()
    print(f"通过 {len(_PASS)} · 失败 {len(_FAIL)} · 未实现 {len(_SKIP)}")
    if _FAIL: print("需要修的：", "、".join(_FAIL))
    if _SKIP: print("还没写的：", "、".join(_SKIP))
    if _PASS and not _FAIL and not _SKIP:
        print("本章全部通过 🎉")

print("自测工具已就绪 —— 每写完一个留白，就运行它下面的测试块。")

### 9.6 实现 `train_epoch`

**要求**：按上面「五个动作」的顺序实现一个 epoch 的训练，并每隔 `log_interval` 打印 loss。

**细节**：
- loss 要除以 `accumulation_steps` 再 backward
- 每步更新学习率：`param_group['lr'] = get_lr(...)`
- 只有累积到 `accumulation_steps` 时才 step + zero_grad
- **混合精度要能在 CPU 上跑**：用 `torch.autocast(device_type=..., enabled=...)`，
  或者设备是 CPU 时用 `contextlib.nullcontext()`

In [ ]:
# ======== 你的实现：train_epoch ========
#
# 【要求】
#   - 签名：train_epoch(epoch, loader, iters, optimizer, scaler, model, args)
#   - 循环里：取 batch -> to(args.device) -> 更新 lr -> 前向 -> loss / accumulation_steps
#   - scaler.scale(loss).backward()
#   - 到累积点时：unscale_ -> clip_grad_norm_(..., args.grad_clip) -> step -> update -> zero_grad
#   - 按 args.log_interval 打印 loss 和 lr
#   - autocast 要能在 CPU 上安全退化（nullcontext 或 enabled=False）
#
# 【可用到的函数 / 方法】
#   - `torch.autocast`、`torch.cuda.amp.GradScaler`、`contextlib.nullcontext`
#   - `torch.nn.utils.clip_grad_norm_`
#   - `optimizer.param_groups`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：train_epoch ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    torch.manual_seed(0)
    from torch import nn, optim
    from torch.utils.data import DataLoader, TensorDataset

    class TinyLM(nn.Module):
        def __init__(self, vocab=16, d=8):
            super().__init__()
            self.emb = nn.Embedding(vocab, d)
            self.head = nn.Linear(d, vocab, bias=False)
        def forward(self, input_ids, labels=None):
            h = self.head(self.emb(input_ids))
            loss = None
            if labels is not None:
                x, y = h[:, :-1, :], labels[:, 1:]
                loss = F.cross_entropy(x.reshape(-1, h.size(-1)), y.reshape(-1), ignore_index=-100)
            return type("Out", (), {"loss": loss})()

    class Args:
        accumulation_steps = 2
        grad_clip = 1.0
        log_interval = 1000
        device = "cpu"
        dtype = "bfloat16"
        learning_rate = 1e-3
        epochs = 1

    model = TinyLM()
    ids = torch.randint(0, 16, (16, 10))
    loader = DataLoader(TensorDataset(ids, ids.clone()), batch_size=4)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3)
    scaler = torch.cuda.amp.GradScaler(enabled=False)

    before = [p.detach().clone() for p in model.parameters()]
    train_epoch(0, loader, len(loader), optimizer, scaler, model, Args())
    after = [p.detach() for p in model.parameters()]
    assert any(not torch.allclose(b, a) for b, a in zip(before, after)), \
        "跑完一个 epoch 参数没变化 —— 检查 optimizer.step() 有没有被调用"

check("train_epoch 能跑通并更新参数", _t)

### 9.7 实现 `get_lr`

把上面演示的余弦调度写成函数。

In [ ]:
# ======== 你的实现：get_lr ========
#
# 【要求】
#   - 签名：get_lr(current_step, total_steps, lr)
#   - 返回 lr * (0.1 + 0.45 * (1 + cos(pi * current_step / total_steps)))
#
# 【可用到的函数 / 方法】
#   - `math.cos`、`math.pi`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：get_lr ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    close(get_lr(0, 100, 5e-4), 5e-4, tol=1e-12, msg="起始学习率应等于 lr")
    close(get_lr(100, 100, 5e-4), 0.1 * 5e-4, tol=1e-12, msg="结束时学习率应衰减到 0.1·lr")
    mid = get_lr(50, 100, 5e-4)
    assert 0.1 * 5e-4 < mid < 5e-4, f"中点学习率应介于两者之间，得到 {mid}"
    assert get_lr(0, 100, 1e-3) > get_lr(50, 100, 1e-3) > get_lr(100, 100, 1e-3), "学习率应单调递减"

check("get_lr 余弦调度", _t)

In [ ]:
summary()

# 第三部分 · 自测与思考

- [ ] 能说清梯度累积为什么能省显存
- [ ] 为什么 loss 要先除以 accumulation_steps？
- [ ] 梯度裁剪应该放在 scaler.step 之前还是之后？（提示：unscale 之后）
- [ ] 余弦调度的起点和终点分别是多少？

**思考题**：如果学习率恒定不衰减，训练后期 loss 会怎样？为什么余弦衰减有效？
